In [ ]:
# ! pip install transformers

In [30]:
# import sys
# !{sys.executable} -m pip install torch --upgrade --force-reinstall

In [31]:
# import sys
# print(sys.executable)

In [ ]:
import torch

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

In [34]:
proxy = "http://cache.ha.univ-nantes.fr:3128/"

os.environ['http_proxy'] = proxy
os.environ['https_proxy'] = proxy
os.environ['HTTP_PROXY'] = proxy
os.environ['HTTPS_PROXY'] = proxy

In [35]:
path = "/home/partage"
print(os.listdir(path))

['Mistral-7B-Instruct-v0.2', 'Llama-2-7b-chat-hf', 'Llama-3.1-8B-Instruct', 'Llama-3.2-1B-Instruct', 'Qwen', 'test.txt', 'Mistral-7B-Instruct-v0.3', 'Meta-Llama-3-8B-Instruct', '.Trash-1001', 'Olmo-3-7B-Instruct', 'microsoft', 'Llama-2-13b-chat-hf', 'google', 'Llama-3.2-3B-Instruct']


In [36]:
MODEL_PATH = path + "/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [37]:
# import ollama
import json

# MODEL_OLLAMA = "Llama-3.2-1B-Instruct"

In [38]:
instructions = ["Improve this paragraph",
                "Improve this paragraph without changing the numbers or units.",
                "Improve this paragraph without changing any facts, figures, units, references, or proper names."]

In [39]:
def generate_improved_text(data, instructions, model):
    """Génère plusieurs réécritures via Ollama"""
    cpt =0
    for article in data:
        cpt += 1
        p1 = article.get("Paragraphes", "")
        for i, instruction in enumerate(instructions):
            prompt = f"""
                        Instruction:
                        {instruction}
                        
                        Paragraph : 
                        {p1}

                        Rewritten paragraph:
                    """
            messages = [
                        {"role": "user", "content": prompt}
                        ]

            inputs = tokenizer.apply_chat_template(
                messages,
                return_tensors="pt",
                add_generation_prompt=True
            )

            input_length = inputs["input_ids"].shape[1]

            outputs = model.generate(
                **inputs,
                pad_token_id=tokenizer.eos_token_id,
                max_new_tokens=300,
                temperature=0.1,
                do_sample=True
            )
            
            response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

            if response.lower().startswith("assistant"):
                response = response[len("assistant"):].strip()
    
            article[f"Rew{i+1}"] = response
            
        if cpt % 25 == 0:
            print(f"======    {cpt} ème paragraphes   ==========")
    return data

In [40]:
with open("../data/json/dataset_simple2.json","r" , encoding='utf-8') as file:
    data = json.load(file)

In [41]:
new_data = generate_improved_text(data, instructions, model)

In [42]:
with open('../data/json/rewrites2.json', 'w', encoding='utf-8') as f:
    json.dump(new_data, f, indent=4, ensure_ascii=False)